In [ ]:

import random
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from tqdm import tqdm
import re


random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

print("Loading GSM8K dataset...")
gsm8k = load_dataset("gsm8k", "main")
train_data = gsm8k["train"]
eval_data = gsm8k["test"]

subset = train_data


model_name = "microsoft/phi-3-mini-4k-instruct"


print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    trust_remote_code=True,
    cache_dir=cache_dir
)
tokenizer.pad_token = tokenizer.eos_token



def preprocess_function(examples):
    inputs = []
    targets = []
    for question, answer in zip(examples["question"], examples["answer"]):
        prompt = f"<|user|>\nSolve this math problem step by step:\n{question}\n<|assistant|>\n"
        inputs.append(prompt)
        targets.append(answer)
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=512, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"].copy()
    for i in range(len(model_inputs["labels"])):
        model_inputs["labels"][i] = [label if label != tokenizer.pad_token_id else -100 for label in model_inputs["labels"][i]]
    return model_inputs


def extract_answer(text):
    numbers = re.findall(r'\d+', text)
    if numbers:
        return numbers[-1]
    return ""

def evaluate_gsm8k(model, tokenizer, eval_dataset, num_samples=100):
    if num_samples < len(eval_dataset):
        eval_subset = eval_dataset.select(range(num_samples))
    else:
        eval_subset = eval_dataset
    correct = 0
    total = 0
    model.eval()
    for example in tqdm(eval_subset, desc="Evaluating"):
        question = example["question"]
        gold_answer = example["answer"]
        gold_number = extract_answer(gold_answer)
        prompt = f"<|user|>\nSolve this math problem step by step:\n{question}\n<|assistant|>\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                num_beams=1,
            )
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_text = generated_text[len(prompt):]
        predicted_number = extract_answer(generated_text)
        if predicted_number == gold_number and gold_number != "":
            correct += 1
        total += 1
    accuracy = correct / total if total > 0 else 0
    return accuracy

# Fine-tune and evaluate
def finetune_and_evaluate(subset, eval_data):
    print(f"\n--- Fine-tuning with provided subset of size: {len(subset)} ---")
    print("Preprocessing data...")
    train_dataset = subset.map(
        preprocess_function,
        batched=True,
        remove_columns=subset.column_names,
        desc="Preprocessing training data"
    )
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )
    training_args = TrainingArguments(
        output_dir="./phi3_gsm8k_custom_subset",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,
        num_train_epochs=3,
        logging_steps=10,
        save_strategy="no",
        fp16=False,
        bf16=True,
        report_to="none",
        push_to_hub=False,
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=1.0,
        warmup_ratio=0.03,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        return_tensors="pt",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    print("Starting fine-tuning...")
    trainer.train()
    print("Evaluating model...")
    accuracy = evaluate_gsm8k(model, tokenizer, eval_data, num_samples=100)
    print(f"Subset size: {len(subset)} | Accuracy: {accuracy:.4f}")
    del model
    torch.cuda.empty_cache()
    return accuracy

# Run the process
accuracy = finetune_and_evaluate(subset, eval_data)
print(f"\nFinal test accuracy on GSM8K: {accuracy:.4f}")


2025-05-05 01:16:07.226259: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-05 01:16:07.306689: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746387967.338948 2542618 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746387967.348145 2542618 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-05 01:16:07.386232: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Loading GSM8K dataset...


README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Could not set the permissions on the file '/home/kundeshwar_pundalik/.cache/huggingface/hub/models--microsoft--phi-3-mini-4k-instruct/blobs/3f311787aa136e858556caa8543015161edcad85ba81b6a36072443d7fa73c87.incomplete'. Error: [Errno 2] No such file or directory: '/home/kundeshwar_pundalik/.cache/huggingface/hub/models--microsoft--phi-3-mini-4k-instruct/tmp_64b85a1e-aa7a-4657-9d3c-baaf58e96d82'.
Continuing without setting permissions.


FileNotFoundError: [Errno 2] No such file or directory: '/home/kundeshwar_pundalik/.cache/huggingface/hub/models--microsoft--phi-3-mini-4k-instruct/blobs/3f311787aa136e858556caa8543015161edcad85ba81b6a36072443d7fa73c87.incomplete'